# Storage

T1 left the workshop running on a flat-priced gas boiler. Now the gas grid offers **time-of-use prices** — cheap at night, expensive during the day — and we add a **thermal tank** that stores heat. The optimizer should run the boiler when gas is cheap, store the heat, and discharge it when gas is expensive.

You'll meet **`Storage`** — a buffer with capacity, charge/discharge efficiency, and self-discharge.

In [ ]:
from datetime import datetime, timedelta

import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from fluxopt import Carrier, Converter, Effect, Flow, Port, Storage, optimize

pio.renderers.default = 'notebook_connected'

## 1. The setup

A 24-hour horizon with cheap nights (0.04 €/kWh, 22:00–06:00) and expensive days (0.12 €/kWh). Heat demand peaks mid-day. Both profiles are plotted below.

In [ ]:
n = 24
timesteps = [datetime(2024, 1, 15) + timedelta(hours=h) for h in range(n)]
gas_price = [0.04 if h < 6 or h >= 22 else 0.12 for h in range(n)]
demand = [
    10,
    10,
    8,
    8,
    8,
    12,
    25,
    60,
    70,
    75,
    75,
    70,
    65,
    60,
    55,
    50,
    45,
    40,
    30,
    25,
    20,
    15,
    12,
    10,
]

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12, subplot_titles=('Heat demand (kW)', 'Gas price (€/kWh)')
)
fig.add_trace(go.Bar(x=timesteps, y=demand, marker_color='#ef553b', showlegend=False), row=1, col=1)
fig.add_trace(
    go.Scatter(x=timesteps, y=gas_price, line_shape='hv', line_color='#636efa', showlegend=False), row=2, col=1
)
fig.update_layout(height=320, margin={'l': 50, 'r': 20, 't': 30, 'b': 20}, template='plotly_white')
fig

## 2. Add a `Storage`

A **`Storage`** has two flows on the same carrier:

- `charging` — energy in (heat from the boiler)
- `discharging` — energy out (heat to the workshop)

Plus three properties that make a storage realistic:

- `capacity` — how much can be stored (MWh)
- `eta_charge` / `eta_discharge` — round-trip efficiency
- `relative_loss_per_hour` — self-discharge (e.g. tank cooldown)

Storage flows attach to the **carrier bus** automatically — no `Port` needed. When both flows share a carrier, the qualified ids become `tank(charge)` and `tank(discharge)`.

In [ ]:
peak = max(demand)

result = optimize(
    timesteps=timesteps,
    carriers=[Carrier('gas'), Carrier('heat')],
    effects=[Effect('cost', unit='EUR')],
    ports=[
        Port(
            'gas_grid',
            imports=[
                Flow('gas', size=1000, effects_per_flow_hour={'cost': gas_price}),
            ],
        ),
        Port(
            'workshop',
            exports=[
                Flow('heat', size=peak, fixed_relative_profile=[d / peak for d in demand]),
            ],
        ),
    ],
    converters=[
        Converter.boiler(
            'boiler',
            thermal_efficiency=0.9,
            fuel_flow=Flow('gas', size=200),
            thermal_flow=Flow('heat', size=120),
        ),
    ],
    storages=[
        Storage(
            'tank',
            capacity=300,
            charging=Flow('heat', size=80),
            discharging=Flow('heat', size=80),
            eta_charge=0.98,
            eta_discharge=0.98,
            relative_loss_per_hour=0.005,
        ),
    ],
    objective_effects='cost',
)
print(f'Total cost: {result.objective:.2f} EUR')

## 3. Read the result

Two new things to look at:

- **Storage level** — `result.storage_level('tank')` returns the energy stored at each timestep (1-D, `time`, in MWh).
- **Storage flows** — they appear in `result.flow_rates` like any other flow, with ids `tank(charge)` and `tank(discharge)`.

In [ ]:
times = result.flow_rates.coords['time'].values
level = result.storage_level('tank')

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12, subplot_titles=('Flow rates', 'Tank level')
)
for fid in result.flow_rates.coords['flow'].values:
    fig.add_trace(
        go.Scatter(x=times, y=result.flow_rate(str(fid)).values, name=str(fid), line_shape='hv'),
        row=1,
        col=1,
    )
fig.add_trace(
    go.Scatter(
        x=level.coords['time'].values, y=level.values, name='level', fill='tozeroy', line_shape='hv', showlegend=False
    ),
    row=2,
    col=1,
)
fig.update_layout(height=420, margin={'l': 50, 'r': 20, 't': 30, 'b': 20}, template='plotly_white')
fig.update_yaxes(title_text='kW', row=1, col=1)
fig.update_yaxes(title_text='kWh', row=2, col=1)
fig

You should see the boiler running through the cheap night hours filling the tank (`tank(charge)` rising, level climbing), then the tank discharging through the expensive midday peak (`tank(discharge)` flowing into the workshop).

The shift only pays when the price gap exceeds the round-trip loss: round-trip efficiency 0.98 × 0.98 ≈ 0.96, plus 0.5 %/h self-discharge, so each stored kWh has to "earn back" roughly 4–10 % of its cost depending on hold time. Try shrinking the price spread or capacity and watch the tank fall idle.

## Recap

You added one new object — `Storage` — to the T1 model. The optimizer figured out the charge/discharge schedule on its own; you only declared the physical reality (capacity, efficiency, losses) and the cost signal (TOU gas price).

**Next**: [Status](03-status.ipynb) — give the boiler an on/off switch with min run time and startup cost.